In [2]:
# San Brown
# sam_brown@mines.edu
# June19
# Goal: Preprocess data and create dataframe for analysis of long-term slip patterns

# Get tide data for entire timeframe using average gz coords (no tide in la stations but timing is still relevant)

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import my_lib.funcs
import Stations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# For this dataset we are focusing on tidal modulation. For each event we will need tide height, tide derivative, form factor, 
# time since last event, slip size (standardized for each station and averaged), high or low tide event, and some indicator
# to signal whether the event is following a skipped low/high tide event.

In [3]:
# Load the paths

df_2008 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2008_2008Events2stas")
df_2009 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2009_2009Events2stas")
df_2010 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2010_2010Events2stas")
df_2011 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2011_2011Events2stas")
df_2012 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2012_2012Events2stas")
df_2013 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas")
df_2014 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2014_2014Events2stas")
df_2015 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2015_2015Events2stas")
df_2016 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2016_2016Events2stas")
df_2017 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2017_2017Events2stas")
df_2018 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2018_2018Events2stas")
df_2019 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2019_2019Events2stas")

# One Large list
all_dfs = (
    df_2008 + df_2009 + df_2010 + df_2011 + df_2012 + df_2013 +
    df_2014 + df_2015 + df_2016 + df_2017 + df_2018 + df_2019
)


In [4]:
# Preprocess
clean_df = my_lib.funcs.extract_event_features(all_dfs)

In [5]:
clean_df[5].head()

,station,pre-slip_area,slip_severity,peak_time,total_delta,start_time
0,la09x,19.045014,1.733345e-07,3435.0,0.089781,2008-11-24 10:35:00
1,slw1x,25.754058,9.762389e-07,3420.0,0.314563,2008-11-24 10:35:00


In [6]:
standards = pd.read_csv('../station_standards.csv')

In [7]:
standards.head(10)

,Station,pre-slip_area,pre-slip_area_sd,slip_severity,slip_severity_sd,slip_size,slip_size_sd
0,la01,120.336973,48.471627,1.026324e-06,3.246978e-07,0.365944,0.057366
1,la02,107.824098,77.972644,1.073755e-06,3.649910e-07,0.377533,0.061455
2,la03,117.900913,93.697779,9.407593e-07,2.875423e-07,0.362463,0.053384
3,la04,89.389549,87.318524,1.017707e-06,3.423616e-07,0.391973,0.065167
4,la05,90.774336,96.847303,8.644231e-07,3.803968e-07,0.358512,0.094409
5,la06,84.062121,79.374442,1.433913e-06,4.613549e-07,0.436199,0.066952
6,la07,103.107350,78.838308,1.011195e-06,3.451209e-07,0.364951,0.059365
7,la08,136.621515,164.461731,5.473297e-07,2.006189e-07,0.253750,0.043152
8,la09,263.442161,300.949402,2.699826e-07,2.097182e-07,0.135288,0.022702
9,la10,62.605822,25.441080,1.250169e-06,4.258395e-07,0.442599,0.079964


In [13]:
# Loop through each event. Loop through each station, append standardized delta in list then average it.
# add start time to data frame

# Initialize dataframe
net_df = pd.DataFrame(columns = ['tide_h', 'tide_deriv', 'form_fac', 'time_since', 'slip_size_standardized', 'high_t_evt', 'start_time'])


for event in clean_df:

    # Initialize list for slip sizes
    slip_deltas = []

    # Loop through rows
    for i, row in event.iterrows():
        station = row['station'][:4]
        if station == 'slw1' or station =='ws04' or station =='ws05':
            continue

        row_sch = standards[standards['Station'] == station]

        # if row_sch.empty:
        #     raise ValueError(f"Station not found in standards: '{station}'")
            
        # Standardization metrics
        station_mean = row_sch['slip_size'].values
        station_sd = row_sch['slip_size_sd'].values[0]

        standardized_val = (row['total_delta'] - station_mean) / station_sd

        slip_deltas.append(standardized_val)

    net_df.loc[len(net_df)] = {
        "slip_size": sum(slip_deltas) / len(slip_deltas),
        "start_time": event.at[0, 'start_time']
    }

In [27]:
# Load tide Data

#Average coordinates for gz stations (source code in severity_class nb)
x_cor = -168955.1491394913 
y_cor = -599694.5432784811

tide_df = my_lib.funcs.get_tide_height(4380, x_cor, y_cor, "2008-01-01 00:00:00") # 12 years worth of data

Elapsed time: 49.69264197349548 seconds


In [32]:
# Organize events by time
net_df = net_df.sort_values('start_time')

In [36]:
# Calculate time since in minutes
net_df['time_since'] = net_df['start_time'].diff().dt.total_seconds() / 60

In [44]:
# Need to get start times down to minutes
tide_df['time'] = tide_df['time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))
net_df['start_time'] = net_df['start_time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))

In [49]:
# Insert Tide values
net_df['start_time'] = pd.to_datetime(net_df['start_time'])
tide_df['time'] = pd.to_datetime(tide_df['time'])

# Merge tide height into net_df based on matching timestamps
merged_df = pd.merge(net_df, tide_df[['time', 'tide_height']], 
                     left_on='start_time', right_on='time', how='left')

# Drop extra 'time' column if you want
merged_df = merged_df.drop(columns=['time'])

In [69]:
# Insert tide derivatives into data
tide_d = my_lib.funcs.tide_derivative(tide_df)
for i, row in merged_df.iterrows():
    time = row['start_time']

    index = tide_d[tide_d['time'] == time].index

    if not index.empty:
        idx = index[0]
        merged_df.at[i, 'tide_deriv'] = tide_d.at[idx, 'tide_deriv']

In [70]:
merged_df.head()

,tide_h,tide_deriv,form_fac,time_since,slip_size,high_t_evt,start_time,tide_height,tide_change
0,-81.877624,-0.268681,NaN,NaN,[-3.054164477665883],NaN,2008-01-25 01:01:00,-81.877624,-0.268681
1,33.714046,0.056003,NaN,1032.50,[-1.8117240153471406],NaN,2008-01-25 18:14:00,33.714046,0.056003
2,5.033164,-0.032735,NaN,1302.75,[-1.1108156912191554],NaN,2008-01-26 15:56:00,5.033164,-0.032735
3,-35.730665,-0.056060,NaN,650.25,[-2.703973582015887],NaN,2008-01-27 02:47:00,-35.730665,-0.056060
4,-3.994414,-0.121354,NaN,797.50,[-1.7047578566323343],NaN,2008-01-27 16:04:00,-3.994414,-0.121354


In [75]:
# Form factor calculation

form_fac = my_lib.funcs.form_factor_calc(tide_df)

/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF/my_lib/funcs.py:411: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide_window, p0=initial_guess)


In [79]:
# Add date-only column to form_fac
form_fac['date_only'] = form_fac['dates'].dt.date

# Loop through each row in avg_dat
for i, event in merged_df.iterrows():
    time = event['start_time']
    target_date = time.date()

    # Select all rows with matching date
    rows_date = form_fac[form_fac['date_only'] == target_date]

    # Compute average form factor for that date
    merged_df.at[i, 'form_fac'] = rows_date['form_factors'].mean()

In [93]:
# Encode high tide vs low tide event
merged_df['high_t_evt'] = (merged_df['tide_h'] > 0).astype(int)

In [98]:
merged_df.head()

,tide_h,tide_deriv,form_fac,time_since,slip_size,high_t_evt,start_time,tide_height
0,-81.877624,-0.268681,1.826057,NaN,[-3.054164477665883],0,2008-01-25 01:01:00,-81.877624
1,33.714046,0.056003,1.826057,1032.50,[-1.8117240153471406],1,2008-01-25 18:14:00,33.714046
2,5.033164,-0.032735,1.560612,1302.75,[-1.1108156912191554],1,2008-01-26 15:56:00,5.033164
3,-35.730665,-0.056060,1.590570,650.25,[-2.703973582015887],0,2008-01-27 02:47:00,-35.730665
4,-3.994414,-0.121354,1.590570,797.50,[-1.7047578566323343],0,2008-01-27 16:04:00,-3.994414


In [100]:
merged_df['slip_size'] = merged_df['slip_size'].str[0]


In [104]:
merged_df.tail()

,tide_h,tide_deriv,form_fac,time_since,slip_size,high_t_evt,start_time,tide_height
5145,NaN,-0.115850,5.939900,1430.0,1.089493,0,2019-11-20 17:54:00,49.657021
5146,NaN,-0.041322,2.461490,800.0,-0.047944,0,2019-11-21 07:14:00,-50.453689
5147,NaN,-0.168629,2.461490,770.0,-0.950877,0,2019-11-21 20:04:00,3.92381
5148,NaN,-0.223637,0.849641,1265.0,0.481421,0,2019-11-22 17:09:00,25.649224
5149,NaN,-0.171789,0.686379,830.0,0.102477,0,2019-11-23 06:59:00,-11.489346


In [106]:
# Export to csv file
merged_df.to_csv('09-18', index = False)